# Mini-project: Applying supervised learning to predict student dropout

**Welcome to your Mini-project: Applying supervised learning to predict student dropout rate**

In this project, we will examine student data and use supervised learning techniques to predict whether a student will drop out. In the education sector, retaining students is vital for the institution's financial stability and for students’ academic success and personal development. A high dropout rate can lead to significant revenue loss, diminished institutional reputation, and lower overall student satisfaction.

You will work with the data in three distinct stages:

1.  Applicant and course information
2.  Student and engagement data
3.  Academic performance data

These stages reflect [Company]’s real-world data journey and how student information has progressed and become available. Additionally, this approach enables you, through data exploration, to support [Company] in better understanding and identifying key metrics to monitor. This approach will also assist you in determining at which stage of the student journey interventions would be most effective.

Please set aside approximately **12 hours** to complete the mini-project.

## Business context
[Company] specialises in providing educational services and resources to students and professionals across various fields. The company's primary focus is on enhancing learning experiences through a range of services, including online courses, tutoring, and educational consulting. By leveraging cutting-edge technology and a team of experienced educators, [Company] aims to bridge the gap between traditional learning methods and the evolving needs of today's learners.

[Company] serves its university partners by establishing strategic partnerships to enhance the universities’ global reach and diversity. It supports the universities in their efforts to attract international students, thereby enriching the cultural and academic landscape of their campuses. It works closely with university faculty and staff to ensure that the universities are prepared and equipped to welcome and support a growing international student body. Its partnership with universities also offers international students a seamless transition into their chosen academic environment.

[Company] runs several International Study Centres across the UK and Dublin in partnership with universities with the aim of preparing a pipeline of talented international students from diverse backgrounds for degree study. These centres help international students adapt to the academic, cultural, and social aspects of studying abroad. This is achieved by improving conversational and subject-specific language skills and academic readiness before students progress to a full degree programme at university.

Through its comprehensive suite of services, it supports learners and universities at every stage of their educational journey, from high school to postgraduate studies. Its approach is tailored to meet the unique needs of each learner, offering personalised learning paths and flexible scheduling options to accommodate various learning styles and commitments.

[Company]'s services are designed to be accessible and affordable, making quality education a reality for many individuals. By focusing on the integration of technology and personalised learning, the company aims to empower learners to achieve their full potential and succeed in their academic and professional pursuits. [Company] is at the forefront of transforming how people learn and grow through its dedication to innovation and excellence.

[Company] has provided you with 3 data sets.


## Objective
By the end of this mini-project, you will have developed the skills and knowledge to apply advanced machine learning techniques to create a predictive model for student dropout. This project will involve comprehensive data exploration, preprocessing, and feature engineering to ensure high-quality input for the models. You will employ and compare multiple predictive algorithms, such as XGBoost, and a neural network-based model, to determine the most effective model for predicting student dropout.

In the Notebook, you will:
- explore the data sets, taking a phased approach
- preprocess the data and conduct feature engineering
- predict the dropout rate using XGBoost, and a neural network-based model.

You will also write a report summarising the results of your findings and recommendations.

## Assessment criteria
By completing this project, you’ll be able to provide evidence that you can:

- develop accurate predictions across diverse organisational scenarios by building and testing advanced ML models
- inform data-driven decision-making with advanced machine learning algorithms and models
- propose and present effective solutions to organisational problems using data preprocessing, model selection, and insightful analysis techniques.

## Project guidance
1.   Navigate to **Mini-project 6.3 Applying supervised learning to predict student dropout**, and save a copy of the activity Notebook to your Drive.

2. Please refer to the Rubric for specific steps to be performed as part of the project activity. Every step mentioned in the rubric will be assessed separately.

3. When you’ve completed the activity:
  - download your completed Notebook as an IPYNB (Jupyter Notebook) or PY (Python) file
  - save the file as follows: **LastName_FirstName_CAM_C201_Week_6_Mini-project**.

4. Prepare a detailed report (800–1,000 words) that includes an overview of your approach, a description of your analysis, and an explanation of the insights you identified - Please refer to the Rubric for further details that should form a part of your analysis and report. Save the document as a PDF named according to the following convention: **LastName_FirstName_CAM_C201_W6_Mini-project.pdf**.

5. Submit your Notebook and PDF document.




# Stage 1 data: Applicant and course information


## **EDA**

In [ ]:
# File URL
stage1 = "<path-to-dataset>"

In [ ]:
# Import the necessary libraries.
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam, RMSprop
import matplotlib.pyplot as plt
import seaborn as sns
from keras.callbacks import EarlyStopping
from keras.regularizers import l2
from sklearn.metrics import roc_auc_score

In [ ]:
# Data exploration
stage1 = pd.read_csv(stage1)
stage1.head()

In [ ]:
stage1.shape

In [ ]:
stage1.info()

Some Boolean data features must be transformed to enable the NN to learn from them.

## Pre-processing instructions
- Remove any columns not useful in the analysis (LearnerCode).
- Remove columns with categorical features with high cardinality (use >200 unique values, as a guideline for this data set).
- Remove columns with > 50% data missing.
- Perform ordinal encoding for ordinal data.
- Perform one-hot encoding for all other categorical data.

In [ ]:
# Remove any columns not useful in the analysis (LearnerCode)
stage1 = stage1.drop('LearnerCode', axis=1)

In [ ]:
stage1.nunique() >200

I decided to add a column to get the age of students before dropping the DOB and the other columns as described on the second and third point on the pre-processing instructions.

In [ ]:
# Transform DOB into Age before drop it
stage1['DateofBirth'] = pd.to_datetime(stage1['DateofBirth'])
stage1['Age'] = 2026 - stage1['DateofBirth'].dt.year

In [ ]:
# Sanity check on Age column
stage1['Age'].describe()

In [ ]:
# Remove columns with categorical features with high cardinality (use >200 unique values, as a guideline for this data set)
high_cardinality = stage1.nunique()[stage1.nunique() > 200].index
stage1.drop(columns=high_cardinality, inplace=True)

In [ ]:
# Remove columns with > 50% data missing.
missing_percentage = stage1.isnull().mean() * 100
missing_columns = missing_percentage[missing_percentage > 50].index
stage1.drop(columns=missing_columns, inplace=True)

In [ ]:
stage1.dtypes

In [ ]:
stage1.shape

In [ ]:
stage1.select_dtypes(include=['object']).columns

In [ ]:
stage1.nunique()

In [ ]:
# Perform ordinal encoding for ordinal data
# Binary mapping for binary categorical columns
binary_mapping = {'Agent': 1, 'Direct': 0, 'Female': 1, 'Male': 0, 'Yes': 1, 'No': 0}

stage1['BookingType'] = stage1['BookingType'].map(binary_mapping)
stage1['Gender'] = stage1['Gender'].map(binary_mapping)
stage1['IsFirstIntake'] = stage1['IsFirstIntake'].astype(int)
stage1['CompletedCourse'] = stage1['CompletedCourse'].map(binary_mapping)

Transformed the features which have 2 observations into binary as mapped above to get it trained by the neural networks.

In [ ]:
stage1.head()

In [ ]:
# Checking the columns that should be enconding for categorical data.
stage1.select_dtypes(include=['object']).columns

In [ ]:
# Perform one-hot encoding for all other categorical data.
stage1 = pd.get_dummies(stage1, columns=['CentreName', 'LeadSource', 'Nationality', 'CourseLevel', 'CourseName', 'ProgressionUniversity'])

In [ ]:
stage1.shape

In [ ]:
stage1.head()

In [ ]:
# Check shape
print("Shape:", stage1.shape)

# Check remaining missing values
print("\nMissing values:\n", stage1.isnull().sum()[stage1.isnull().sum() > 0])

# Check data types
print("\nData types:\n", stage1.dtypes)

# Check target variable
print("\nTarget variable distribution:\n", stage1['CompletedCourse'].value_counts())

# Check IsFirstIntake is binary
print("\nIsFirstIntake unique values:", stage1['IsFirstIntake'].unique())

In [ ]:
# Turn the bool data types to figures
stage1 = stage1.astype({col: int for col in stage1.select_dtypes(bool).columns})

In [ ]:
print(stage1.dtypes.value_counts())

## Fill in any missing values in the remaining columns.

In [ ]:
# Missing values
missing_percentage = stage1.isnull().mean() * 100
missing_columns = missing_percentage[missing_percentage > 0].index
missing_columns

In [ ]:
# Fill the missing values with mean
stage1.fillna(stage1.mean(), inplace=True)

In [ ]:
# Fill missing values to IsFirstIntake column
stage1['IsFirstIntake'].fillna(0, inplace=True)

In [ ]:
# evaluate missing values fix
stage1.isnull().sum()

In [ ]:
stage1.shape

## Define features and a target variable.

In [ ]:
# Define features and target varible
y = stage1['CompletedCourse']
X = stage1.drop('CompletedCourse', axis=1)

In [ ]:
# Plot target variable
sns.countplot(x='CompletedCourse', data=stage1)
plt.title('Target Variable Distribution')
plt.show()

The plot above, based on the Course Completed criterion and the data, shows an imbalance.

There is a business/domain-expert decision to drop additional columns that may be irrelevant to the dataset. For example, Nationality or Course Level. From the analyst's perspective, nationality can introduce bias, but it will be kept to a minimum to enhance potential business acumen.

## Train test

In [ ]:
# Split the data into training and test sets 80/20
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Validation dataset 10%
X_train_1, X_val_1, y_train_1, y_val_1 = train_test_split(X_train_1, y_train_1, test_size=0.1, random_state=42)

## XGBoost

In [ ]:
# XGBOOST model and fitted on the training dataset
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train_1, y_train_1)

In [ ]:
# Model evaluation - Model 1
y_pred_1 = xgb_model.predict(X_test_1)
print("Accuracy:", accuracy_score(y_test_1, y_pred_1))
print("Confusion Matrix:", confusion_matrix(y_test_1, y_pred_1))
print("Precision:", precision_score(y_test_1, y_pred_1))
print("Recall:", recall_score(y_test_1, y_pred_1))

cm = confusion_matrix(y_test_1, y_pred_1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Model 1')

The Accuracy for the base model is 89%, and the Recall is 95%, where the precision is 92%.
The Confusion Matrix resulted in 356 false positives, which could mean students were flagged for intervention unnecessarily. Also there are 186 false negatives that could be concerned.

In [ ]:
# AUC plot
from sklearn.metrics import roc_curve, auc
y_prob_1 = xgb_model.predict_proba(X_test_1)[:,1]

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test_1, y_prob_1)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Model 1')
plt.legend()
plt.show()

print("AUC Score:", roc_auc)

The ROC curve score is 0.875 (87.5%), indicating that 87.5% of the dataset was correctly classified.

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({'Feature': X_train_1.columns, 'Importance': xgb_model.feature_importances_})
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)
feature_importance = feature_importance.head(20)
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance)
plt.title('Feature Importance')
plt.show()

The data indicate that Nationality is the most important predictor of dropout, aligning with the [Company]'s focus on international students and suggesting that targeted support based on students' country of origin, followed by Course Level Foundation. It may indicate opportunities to change the process, if needed, to avoid student dropout.

In [ ]:
# XGBoost hyperparameter
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5]
}
grid_search = GridSearchCV(XGBClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_1, y_train_1)
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

In [ ]:
# Best parameters
print("Best parameters Stage 2:", grid_search.best_params_)

In [ ]:
# Best model with the best parameters tuned
y_pred_1_tuned = best_model.predict(X_test_1)
print("Accuracy:", accuracy_score(y_test_1, y_pred_1_tuned))
print("Confusion Matrix:", confusion_matrix(y_test_1, y_pred_1_tuned))
print("Precision:", precision_score(y_test_1, y_pred_1_tuned))
print("Recall:", recall_score(y_test_1, y_pred_1_tuned))

cm = confusion_matrix(y_test_1, y_pred_1_tuned)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Model 2')

On Model 2, the Accuracy is similar to Model 1 at 89%, as well as the Recall, 95%, and the precision is slightly better, 92%.
The Confusion Matrix resulted in 351 false positives and 186 false negatives on the dataset.

In [ ]:
# AUC plot
y_prob_1_tuned = best_model.predict_proba(X_test_1)[:,1]

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test_1, y_prob_1_tuned)
roc_auc_1_tuned = auc(fpr, tpr)

# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.1f)' % roc_auc_1_tuned)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Model 2')
plt.legend()
plt.show()

print("AUC Score:", roc_auc_1_tuned)

The ROC curve score is 0.877 (87.7%), indicating that 87.7% of the dataset was correctly classified a results that is slightly better for the Model 2 Tuned in contrast to the Model 1.

In [ ]:
# Compared both XGBoost Models
print("XGBoost Model 1")
print("Accuracy:", accuracy_score(y_test_1, y_pred_1))
print("Precision:", precision_score(y_test_1, y_pred_1))
print("Recall:", recall_score(y_test_1, y_pred_1))
print("AUC:", roc_auc)

print("XGBoost Model 2 (Tuned)")
print("Accuracy:", accuracy_score(y_test_1, y_pred_1_tuned))
print("Precision:", precision_score(y_test_1, y_pred_1_tuned))
print("Recall:", recall_score(y_test_1, y_pred_1_tuned))
print("AUC:", roc_auc_1_tuned)

In [ ]:
# Plot visualization for comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'AUC']
original_scores = [
    accuracy_score(y_test_1, y_pred_1),
    precision_score(y_test_1, y_pred_1),
    recall_score(y_test_1, y_pred_1),
    roc_auc
]
tuned_scores = [
    accuracy_score(y_test_1, y_pred_1_tuned),
    precision_score(y_test_1, y_pred_1_tuned),
    recall_score(y_test_1, y_pred_1_tuned),
    roc_auc_1_tuned
]

# Plot
x = range(len(metrics))
fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x, original_scores, width=0.4, label='Original', align='center')
bars2 = ax.bar([i + 0.4 for i in x], tuned_scores, width=0.4, label='Tuned', align='center')

# Add data labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks([i + 0.2 for i in x])
ax.set_xticklabels(metrics)
ax.set_title('XGBoost Stage 1: Original vs Tuned')
ax.set_ylabel('Score')
ax.set_ylim(0.8, 1.0)
ax.legend()
plt.show()

In [ ]:
# Results table XGBoost Stage 1
xgb_stage1_results = pd.DataFrame({
    'Model': ['XGBoost Model 1 (Baseline)', 'XGBoost Model 2 (Tuned)'],
    'Accuracy': [
        round(accuracy_score(y_test_1, y_pred_xgb_stage1), 4),
        round(accuracy_score(y_test_1, y_pred_xgb_tuned_stage1), 4)
    ],
    'Precision': [
        round(precision_score(y_test_1, y_pred_xgb_stage1), 4),
        round(precision_score(y_test_1, y_pred_xgb_tuned_stage1), 4)
    ],
    'Recall': [
        round(recall_score(y_test_1, y_pred_xgb_stage1), 4),
        round(recall_score(y_test_1, y_pred_xgb_tuned_stage1), 4)
    ],
    'AUC': [
        round(roc_auc_score(y_test_1, y_prob_xgb_stage1), 4),
        round(roc_auc_score(y_test_1, y_prob_xgb_tuned_stage1), 4)
    ],
    'True Negatives': [
        confusion_matrix(y_test_1, y_pred_xgb_stage1)[0][0],
        confusion_matrix(y_test_1, y_pred_xgb_tuned_stage1)[0][0]
    ],
    'False Positives': [
        confusion_matrix(y_test_1, y_pred_xgb_stage1)[0][1],
        confusion_matrix(y_test_1, y_pred_xgb_tuned_stage1)[0][1]
    ],
    'False Negatives': [
        confusion_matrix(y_test_1, y_pred_xgb_stage1)[1][0],
        confusion_matrix(y_test_1, y_pred_xgb_tuned_stage1)[1][0]
    ],
    'True Positives': [
        confusion_matrix(y_test_1, y_pred_xgb_stage1)[1][1],
        confusion_matrix(y_test_1, y_pred_xgb_tuned_stage1)[1][1]
    ]
})

# Set Model as index
xgb_stage1_results = xgb_stage1_results.set_index('Model')

# Apply heatmap styling
xgb_stage1_results.style.background_gradient(cmap='RdYlGn', axis=0)

The XGBoost model methodology is great at distinguishing between dropouts and non-dropouts. Two models were tested, and the second was slightly better.

Model 1 - Original
Accuracy: 89.1%
Precision: 91.9%
Recall: 95.6%
AUC: 87.2%

Model 2 - Tuned
Accuracy: 89.2%
Precision: 92%
Recall: 95.6%
AUC: 87.5%


The first one, the Model 1 used as a baseline, has an accuracy of 89.1%, the Precision of 91.9%, and the Recall of 95.6% on the actual dropouts. The ROC and AUC scores are 87.2%, which means we correctly classified a bit over 87% of the observations.

In the Confusion Matrix, the 356 false positives could mean students were unnecessarily flagged for intervention. More details from the domain expert would be necessary to feed the dataset. There were 186 identified missed dropouts.

Model 2, the Tuned one, reached an accuracy of 89.2%, the Precision of 92%, and the Recall of 95.6%. The ROC and AUC scores are 87.5%, which means we correctly classified a bit better in this model than in the first one.

Looking at the Confusion Matrix, there are 353 false positives, which is better than the previous model. The model identified 184 missed dropouts as the most concerning for the [Company].

As the data is imbalanced, the model can oversample; therefore, it can be biased towards the majority class (students who didn't dropout) and overestimate the model performance, which means it can struggle to correctly identify students who are at risk of dropping out and impact the overall results.

## Neural Network

**Defining the functions**

In [ ]:
# Standarized the features
scaler = StandardScaler()
X_train_1_scaled = scaler.fit_transform(X_train_1)
X_val_1_scaled = scaler.transform(X_val_1)
X_test_1_scaled = scaler.transform(X_test_1)

In [ ]:
# Number of features
num_features_stage1 = X_train_1_scaled.shape[1]

In [ ]:
# Print number of features
print("Number of features:", num_features_stage1)

In [ ]:
# Import Tensorflow
import tensorflow as tf

# One input layer of a number of neurons equal to the number of input features
model_nn_stage1 = tf.keras.Sequential([tf.keras.layers.Dense(32, activation='relu', input_shape=(X_train_1_scaled.shape[1],)),
                             tf.keras.layers.Dense(16, activation='relu'),
                             tf.keras.layers.Dense(1, activation='sigmoid')])


In [ ]:
# Binary cross entropy as loss
model_nn_stage1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

Used Adam as the dataset is tabular and structured and used to fit better.
As the result is binary (dropout/not dropout) I decided to use binary cross entropy as well.

**Create Neural Network and train the models**

In [ ]:
# Create and train models with Adam
adam_stage1 = model_nn_stage1.fit(X_train_1_scaled, y_train_1, epochs=50, batch_size=32, validation_data=(X_val_1_scaled, y_val_1))

Learned all 50 epochs consistently.

In [ ]:
# Loss curves for each epoch plotted
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(adam_stage1.history['loss'], label='Training Loss')
plt.plot(adam_stage1.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss - Model 1')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(adam_stage1.history['accuracy'], label='Training Accuracy')
plt.plot(adam_stage1.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy - Model 1')
plt.legend()

plt.tight_layout()
plt.show()

It's clear that the Model 1 is overfitting, as the gap between the training and validation losses is large.

Training Accuracy starts lower, but it's expected since the first epoch takes a bit longer to learn from the dataset. However, on epoch 9, the validation accuracy started to decrease.

**Function with dropout**

In [ ]:
# Function regularation and dropout
def build_model(optimizer):
    model_stage1 = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu',
                              input_shape=(X_train_1_scaled.shape[1],),
                              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.Dense(16, activation='relu',
                              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model_stage1.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model_stage1

In [ ]:
# Adam optimization and L2 regularisation
adam_stage1_l2 = build_model(Adam(learning_rate=0.001))
adam_stage1_l2_history = adam_stage1_l2.fit(X_train_1_scaled, y_train_1, epochs=50, batch_size=32, validation_data=(X_val_1_scaled, y_val_1))

In [ ]:
# Sanity check
print(np.isnan(X_train_1_scaled).sum())
print(np.isnan(X_test_1_scaled).sum())

In [ ]:
# Loss curves for each epoch plotted
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(adam_stage1_l2_history.history['loss'], label='Training Loss')
plt.plot(adam_stage1_l2_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss - Model 2')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(adam_stage1_l2_history.history['accuracy'], label='Training Accuracy')
plt.plot(adam_stage1_l2_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy - Model 2')
plt.legend()

plt.tight_layout()
plt.show()

Traning loss dropped a bit and it stabilies compred to the first NN model. Validation loss is also more stable than before and the gab between them reduced in contrast to the first model.

Traning and validation accuracy is much closer both are hovering around 90% and the lines are less divergent.

However despite the improvements to reduce the overfitting there are some issues as validation loss incresed after epoch 10 and the validation loss is still unstable.

**Early stopping**

In [ ]:
# Neural layers
def build_model(optimizer):
  adam_stage1_l2_es = tf.keras.Sequential([tf.keras.layers.Dense(32, activation='relu', input_shape=(X_train_1_scaled.shape[1],), kernel_regularizer=l2(0.001)),
                               tf.keras.layers.Dropout(0.3),
                             tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
                               tf.keras.layers.Dropout(0.3),
                             tf.keras.layers.Dense(1, activation='sigmoid')])
  adam_stage1_l2_es.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

  return adam_stage1_l2_es

# Early stopping
early_stopping_stage1 = tf.keras.callbacks.EarlyStopping(
      monitor="val_loss", patience=5, verbose=0, mode="auto", baseline=None, restore_best_weights=True)

# Pass it to the model
adam_stage1_l2_es = build_model(Adam(learning_rate=0.001))
adam_stage1_l2_es_history = adam_stage1_l2_es.fit(X_train_1_scaled, y_train_1, epochs=50, batch_size=32, validation_data=(X_val_1_scaled, y_val_1), callbacks=[early_stopping_stage1])

The train model stops at epoch 24.

In [ ]:
# Loss curves for each epoch plotted
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(adam_stage1_l2_es_history.history['loss'], label='Training Loss')
plt.plot(adam_stage1_l2_es_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss - Model 3')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(adam_stage1_l2_es_history.history['accuracy'], label='Training Accuracy')
plt.plot(adam_stage1_l2_es_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy - Model 3')
plt.legend()

plt.tight_layout()
plt.show()

The train and validation loss gets down on, the curves are closer, and no more divergence.
On the accuracy side curves gets more stable and the accuracy is slitly higher than the validation. Both are near 90%.

The model reduced overfitting and is the best choice so far. Knowing that, we are going to try different optimizers, activiation functions to test if it can get better.

**Test with one extra layer**

In [ ]:
# Add 64 layer as test to Adam optimizer
def build_model_adam_stage1_v2(optimizer):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train_1_scaled.shape[1],), kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

early_stopping2_stage1 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

adam_stage1_l2_es2 = build_model_adam_stage1_v2(Adam(learning_rate=0.001))
adam_stage1_l2_es2_history = adam_stage1_l2_es2.fit(X_train_1_scaled, y_train_1, epochs=50, batch_size=32, validation_data=(X_val_1_scaled, y_val_1), callbacks=[early_stopping2_stage1])

With an extra layer, the epoch stopped on the 18.

In [ ]:
# Loss curves for each epoch plotted - Model 4
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(adam_stage1_l2_es2_history.history['loss'], label='Training Loss')
plt.plot(adam_stage1_l2_es2_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss - Model 4')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(adam_stage1_l2_es2_history.history['accuracy'], label='Training Accuracy')
plt.plot(adam_stage1_l2_es2_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy - Model 4')
plt.legend()

plt.tight_layout()
plt.show()

Model 4 started to reduce the training on epoch 5 on epoch 6 the validation loss jumped to 0.3164 and went down since the epoch 6.
Training and Validation Accuracy become unstable until training stops at epoch 18.

**Test with RMSprop**

In [ ]:
# RMSprop with the same number of layers - Model 5
def build_model_rmsprop_stage1_v1(optimizer):
    rmsprop_stage1_l2_es = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(X_train_1_scaled.shape[1],), kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    rmsprop_stage1_l2_es.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return rmsprop_stage1_l2_es

early_stopping_rmsprop_stage1 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

rmsprop_stage1_l2_es = build_model_rmsprop_stage1_v1(RMSprop(learning_rate=0.001))
rmsprop_stage1_l2_es_history = rmsprop_stage1_l2_es.fit(
    X_train_1_scaled, y_train_1, epochs=50, batch_size=32,
    validation_data=(X_val_1_scaled, y_val_1), callbacks=[early_stopping_rmsprop_stage1])

RMSprop stopped on 7 epochs which means the model didn't learn enough compared to Adam optimizer.

In [ ]:
# Loss curves for each epoch plotted
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(rmsprop_stage1_l2_es_history.history['loss'], label='Training Loss')
plt.plot(rmsprop_stage1_l2_es_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss - RMSprop - Model 5')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(rmsprop_stage1_l2_es_history.history['accuracy'], label='Training Accuracy')
plt.plot(rmsprop_stage1_l2_es_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy - RMSprop - Model 5')
plt.legend()

plt.tight_layout()
plt.show()

The RMSprop showed to be unstable and slowly increasing.

In [ ]:
# Results table Neural Network Stage 1
nn_stage1_results = pd.DataFrame({
    'Model': [
        'NN Model 1 (Baseline)',
        'NN Model 2 (L2)',
        'NN Model 3 (Adam 32/16 + ES)',
        'NN Model 4 (Adam 64/32/16 + ES)',
        'NN Model 5 (RMSprop 32/16 + ES)'
    ],
    'Accuracy': [
        round(accuracy_score(y_test_1, y_pred_nn_baseline_stage1), 4),
        round(accuracy_score(y_test_1, y_pred_nn_l2_stage1), 4),
        round(accuracy_score(y_test_1, y_pred_nn_v1_stage1), 4),
        round(accuracy_score(y_test_1, y_pred_nn_v2_stage1), 4),
        round(accuracy_score(y_test_1, y_pred_nn_rmsprop_stage1), 4)
    ],
    'Precision': [
        round(precision_score(y_test_1, y_pred_nn_baseline_stage1), 4),
        round(precision_score(y_test_1, y_pred_nn_l2_stage1), 4),
        round(precision_score(y_test_1, y_pred_nn_v1_stage1), 4),
        round(precision_score(y_test_1, y_pred_nn_v2_stage1), 4),
        round(precision_score(y_test_1, y_pred_nn_rmsprop_stage1), 4)
    ],
    'Recall': [
        round(recall_score(y_test_1, y_pred_nn_baseline_stage1), 4),
        round(recall_score(y_test_1, y_pred_nn_l2_stage1), 4),
        round(recall_score(y_test_1, y_pred_nn_v1_stage1), 4),
        round(recall_score(y_test_1, y_pred_nn_v2_stage1), 4),
        round(recall_score(y_test_1, y_pred_nn_rmsprop_stage1), 4)
    ],
    'AUC': [
        round(roc_auc_score(y_test_1, y_prob_nn_baseline_stage1), 4),
        round(roc_auc_score(y_test_1, y_prob_nn_l2_stage1), 4),
        round(roc_auc_score(y_test_1, y_prob_nn_v1_stage1), 4),
        round(roc_auc_score(y_test_1, y_prob_nn_v2_stage1), 4),
        round(roc_auc_score(y_test_1, y_prob_nn_rmsprop_stage1), 4)
    ],
    'True Negatives': [
        confusion_matrix(y_test_1, y_pred_nn_baseline_stage1)[0][0],
        confusion_matrix(y_test_1, y_pred_nn_l2_stage1)[0][0],
        confusion_matrix(y_test_1, y_pred_nn_v1_stage1)[0][0],
        confusion_matrix(y_test_1, y_pred_nn_v2_stage1)[0][0],
        confusion_matrix(y_test_1, y_pred_nn_rmsprop_stage1)[0][0]
    ],
    'False Positives': [
        confusion_matrix(y_test_1, y_pred_nn_baseline_stage1)[0][1],
        confusion_matrix(y_test_1, y_pred_nn_l2_stage1)[0][1],
        confusion_matrix(y_test_1, y_pred_nn_v1_stage1)[0][1],
        confusion_matrix(y_test_1, y_pred_nn_v2_stage1)[0][1],
        confusion_matrix(y_test_1, y_pred_nn_rmsprop_stage1)[0][1]
    ],
    'False Negatives': [
        confusion_matrix(y_test_1, y_pred_nn_baseline_stage1)[1][0],
        confusion_matrix(y_test_1, y_pred_nn_l2_stage1)[1][0],
        confusion_matrix(y_test_1, y_pred_nn_v1_stage1)[1][0],
        confusion_matrix(y_test_1, y_pred_nn_v2_stage1)[1][0],
        confusion_matrix(y_test_1, y_pred_nn_rmsprop_stage1)[1][0]
    ],
    'True Positives': [
        confusion_matrix(y_test_1, y_pred_nn_baseline_stage1)[1][1],
        confusion_matrix(y_test_1, y_pred_nn_l2_stage1)[1][1],
        confusion_matrix(y_test_1, y_pred_nn_v1_stage1)[1][1],
        confusion_matrix(y_test_1, y_pred_nn_v2_stage1)[1][1],
        confusion_matrix(y_test_1, y_pred_nn_rmsprop_stage1)[1][1]
    ]
})

# Set Model as index
nn_stage1_results = nn_stage1_results.set_index('Model')

# Apply heatmap styling
nn_stage1_results.style.background_gradient(cmap='RdYlGn', axis=0)

For the Neural Networks, Model 3 (Adam 32/16) had great results, achieving the highest Accuracy (0.894), Recall (0.955), and AUC (0.866) while minimising missed dropouts (188 False Negatives).

## Comparison between models Stage 1

In [ ]:
# Model Comparison - Stage 1
# XGBoost predictions - Model 1
y_pred_xgb_stage1 = xgb_model.predict(X_test_1)
y_prob_xgb_stage1 = xgb_model.predict_proba(X_test_1)[:,1]

# XGBoost predictions - Model 2 tuned
y_pred_xgb_tuned_stage1 = best_model.predict(X_test_1)
y_prob_xgb_tuned_stage1 = best_model.predict_proba(X_test_1)[:,1]

# Neural Network predictions - All 5 models
y_pred_nn_baseline_stage1 = (model_nn_stage1.predict(X_test_1_scaled) > 0.5).astype(int)
y_prob_nn_baseline_stage1 = model_nn_stage1.predict(X_test_1_scaled)

y_pred_nn_l2_stage1 = (adam_stage1_l2.predict(X_test_1_scaled) > 0.5).astype(int)
y_prob_nn_l2_stage1 = adam_stage1_l2.predict(X_test_1_scaled)

y_pred_nn_v1_stage1 = (adam_stage1_l2_es.predict(X_test_1_scaled) > 0.5).astype(int)
y_prob_nn_v1_stage1 = adam_stage1_l2_es.predict(X_test_1_scaled)

y_pred_nn_v2_stage1 = (adam_stage1_l2_es2.predict(X_test_1_scaled) > 0.5).astype(int)
y_prob_nn_v2_stage1 = adam_stage1_l2_es2.predict(X_test_1_scaled)

y_pred_nn_rmsprop_stage1 = (rmsprop_stage1_l2_es.predict(X_test_1_scaled) > 0.5).astype(int)
y_prob_nn_rmsprop_stage1 = rmsprop_stage1_l2_es.predict(X_test_1_scaled)

# All models dictionary
models_stage1 = {
    'XGBoost_stage1': (y_pred_xgb_stage1, y_prob_xgb_stage1),
    'XGBoost_Tuned_stage1': (y_pred_xgb_tuned_stage1, y_prob_xgb_tuned_stage1),
    'NN_Baseline_stage1': (y_pred_nn_baseline_stage1, y_prob_nn_baseline_stage1),
    'NN_L2_stage1': (y_pred_nn_l2_stage1, y_prob_nn_l2_stage1),
    'NN_Adam_32_16_stage1': (y_pred_nn_v1_stage1, y_prob_nn_v1_stage1),
    'NN_Adam_64_32_16_stage1': (y_pred_nn_v2_stage1, y_prob_nn_v2_stage1),
    'NN_RMSprop_32_16_stage1': (y_pred_nn_rmsprop_stage1, y_prob_nn_rmsprop_stage1)
}

for name, (y_pred, y_prob) in models_stage1.items():
    print(f"\n=== {name} ===")
    print("Accuracy:", accuracy_score(y_test_1, y_pred))
    print("Precision:", precision_score(y_test_1, y_pred))
    print("Recall:", recall_score(y_test_1, y_pred))
    print("AUC:", roc_auc_score(y_test_1, y_prob))
    print("Confusion Matrix:\n", confusion_matrix(y_test_1, y_pred))

In [ ]:
# Plot comparison of all models Stage 1 - Horizontal
metrics_names = ['Accuracy', 'Precision', 'Recall', 'AUC']

# Calculate scores for each model
scores_stage1 = {}
for name, (y_pred, y_prob) in models_stage1.items():
    scores_stage1[name] = [
        accuracy_score(y_test_1, y_pred),
        precision_score(y_test_1, y_pred),
        recall_score(y_test_1, y_pred),
        roc_auc_score(y_test_1, y_prob)
    ]

fig, axes = plt.subplots(1, 4, figsize=(20, 6))

for idx, metric in enumerate(metrics_names):
    ax = axes[idx]
    model_names = list(scores_stage1.keys())
    scores = [scores_stage1[m][idx] for m in model_names]

    bars = ax.barh(model_names, scores, color='#2171b5')

    # Add data labels
    for bar, score in zip(bars, scores):
        ax.text(bar.get_width() - 0.002, bar.get_y() + bar.get_height()/2,
                f'{score:.3f}', ha='right', va='center',
                fontsize=8, color='white', fontweight='bold')

    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_xlim(0.8, 1.0)
    ax.set_xlabel('Score')
    ax.grid(True, alpha=0.3)

plt.suptitle('Model Comparison - Stage 1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Between the two methodologies, we had mixed results as the XGBoost Model 2 Tuned reached higher accuracy (0.892) and AUC (0.877). However, if we look at Precision, Model 4 (0.928) performed better in the Neural Network. Since the data is imbalanced the Recall would be the primary metric was Model 1 (0.957) of the Neural Networks.

# Stage 2 data: Student engagement data

## EDA

In [ ]:
# File URL
stage2 = "<path-to-dataset>"

In [ ]:
# Import the necessary libraries.
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam, RMSprop
import matplotlib.pyplot as plt
import seaborn as sns
from keras.callbacks import EarlyStopping
from keras.regularizers import l2
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.metrics import roc_curve, auc
import tensorflow as tf

In [ ]:
# Data exploration
stage2 = pd.read_csv(stage2)
stage2.head()

In [ ]:
stage2.shape

In [ ]:
stage2.info()

In [ ]:
# Missing values
stage2.isnull().sum()

## Pre-processing instructions

- Remove any columns not useful in the analysis (LearnerCode).
- Remove columns with categorical features with high cardinality (use >200 unique values, as a guideline for this data set).
- Remove columns with >50% data missing.
- Perform ordinal encoding for ordinal data.
- Perform one-hot encoding for all other categorical data.
- Choose how to engage with missing values, which can be done in one of two ways for this project:
  *   Impute the rows with appropriate values.
  *   Remove rows with missing values but ONLY in cases where rows with missing values are minimal: <2% of the overall data.



In [ ]:
# Remove any columns not useful in the analysis (LearnerCode)
stage2 = stage2.drop('LearnerCode', axis=1)

In [ ]:
# Transform DOB into Age before drop it
stage2['DateofBirth'] = pd.to_datetime(stage2['DateofBirth'])
stage2['Age'] = 2026 - stage2['DateofBirth'].dt.year

In [ ]:
# Sanity check on Age column
stage2['Age'].describe()

In [ ]:
# Remove columns with categorical features with high cardinality (use >200 unique values, as a guideline for this data set)
high_cardinality_2 = stage2.nunique()[stage2.nunique() > 200].index
stage2.drop(columns=high_cardinality_2, inplace=True)

In [ ]:
# Remove columns with > 50% data missing.
missing_percentage_2 = stage2.isnull().mean() * 100
missing_columns_2 = missing_percentage_2[missing_percentage_2 > 50].index
stage2.drop(columns=missing_columns_2, inplace=True)

In [ ]:
stage2.dtypes

In [ ]:
stage2.shape

In [ ]:
stage2.nunique()

In [ ]:
print(stage2['BookingType'].unique())
print(stage2['Gender'].unique())
print(stage2['CompletedCourse'].unique())
print(stage2['IsFirstIntake'].unique())

In [ ]:
# Perform ordinal encoding for ordinal data
# Binary mapping for binary categorical columns
binary_mapping2 = {'Agent': 1, 'Direct': 0, 'Female': 1, 'Male': 0, 'Yes': 1, 'No': 0}

stage2['BookingType'] = stage2['BookingType'].map(binary_mapping2)
stage2['Gender'] = stage2['Gender'].map(binary_mapping2)
stage2['IsFirstIntake'] = stage2['IsFirstIntake'].astype(int)
stage2['CompletedCourse'] = stage2['CompletedCourse'].map(binary_mapping2)

In [ ]:
# Sanity check on the binary data applied
print(stage2['BookingType'].unique())
print(stage2['Gender'].unique())
print(stage2['IsFirstIntake'].unique())
print(stage2['CompletedCourse'].unique())

In [ ]:
# Checking the columns that should be enconding for categorical data.
stage2.select_dtypes(include=['object']).columns

In [ ]:
# Perform one-hot encoding for all other categorical data.
stage2 = pd.get_dummies(stage2, columns=['CentreName', 'LeadSource', 'Nationality', 'CourseLevel', 'CourseName', 'ProgressionUniversity'])

In [ ]:
stage2.shape

In [ ]:
# Check high cardinality columns dropped
print("High cardinality columns dropped:", high_cardinality_2.tolist())

# Check missing columns dropped
print("Missing columns dropped:", missing_columns_2.tolist())

One of the aditional columns for this stage is gone, Unauthorised Absence Count.

In [ ]:
stage2.head()

In [ ]:
# Check percentage of missing values for each column
missing_percentage_2 = stage2.isnull().sum() / len(stage2) * 100
print(missing_percentage_2[missing_percentage_2 > 0])

In [ ]:
print(len(stage2))
print(stage2['AuthorisedAbsenceCount'].isnull().sum())

The total number of rows missing is 208 which means it could be dropped however this is one of the columns, Authorised Absence Count, that could impact the learnings from this stage. It was decided to calculate the mean for the missing values.

In [ ]:
# Calculate the mean to missing columns
stage2['AuthorisedAbsenceCount'] = stage2['AuthorisedAbsenceCount'].fillna(stage2['AuthorisedAbsenceCount'].mean())

In [ ]:
# Sanity check of the missing columns
print(stage2['AuthorisedAbsenceCount'].isnull().sum())

In [ ]:
print("Shape:", stage2.shape)
print("\nMissing values:\n", stage2.isnull().sum()[stage2.isnull().sum() > 0])
print("\nData types:\n", stage2.dtypes)
print("\nTarget variable distribution:\n", stage2['CompletedCourse'].value_counts())

In [ ]:
stage2 = stage2.astype({col: int for col in stage2.select_dtypes(bool).columns})

In [ ]:
print(stage2.dtypes.value_counts())

## Define features and a target variable

In [ ]:
# Define features and target varible
y = stage2['CompletedCourse']
X = stage2.drop('CompletedCourse', axis=1)

In [ ]:
sns.countplot(x='CompletedCourse', data=stage2)
plt.title('Target Variable Distribution Stage 2')
plt.show()

The Stage 2 dataset keeps imbalanced as the Stage 1.

## Train test

In [ ]:
# 80/20 split
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 10% validation split
X_train_2, X_val_2, y_train_2, y_val_2 = train_test_split(X_train_2, y_train_2, test_size=0.1, random_state=42)

## XGBoost

In [ ]:
# XGBoost Stage 2
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

xgb_model_stage2 = XGBClassifier(random_state=42)
xgb_model_stage2.fit(X_train_2, y_train_2)

In [ ]:
# Model evaluation Stage 2
y_pred_xgb_stage2 = xgb_model_stage2.predict(X_test_2)
print("Accuracy:", accuracy_score(y_test_2, y_pred_xgb_stage2))
print("Confusion Matrix:", confusion_matrix(y_test_2, y_pred_xgb_stage2))
print("Precision:", precision_score(y_test_2, y_pred_xgb_stage2))
print("Recall:", recall_score(y_test_2, y_pred_xgb_stage2))

cm = confusion_matrix(y_test_2, y_pred_xgb_stage2)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix Stage 2 - Model 1')
plt.show()

The Accuracy for the base model is 90.2%, and the Recall is 96.1%, where the precision is 92.6%. The Confusion Matrix showed 325 false positives, which could mean students were unnecessarily flagged for intervention. Also, there are 164 false negatives that could be of concern and the main focus from the business perspective.

In [ ]:
# AUC plot Stage 2
y_prob_xgb_stage2 = xgb_model_stage2.predict_proba(X_test_2)[:,1]

# Calculate ROC curve
fpr_2, tpr_2, thresholds_2 = roc_curve(y_test_2, y_prob_xgb_stage2)
roc_auc_xgb_stage2 = auc(fpr_2, tpr_2)

# Plot ROC curve
plt.figure()
plt.plot(fpr_2, tpr_2, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc_xgb_stage2)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Stage 2 - Model 1')
plt.legend()
plt.show()

print("AUC Score:", roc_auc_xgb_stage2)

The accuracy score is really good for the first model, with 90%.


In [ ]:
# Feature importance Stage 2
feature_importance_2 = pd.DataFrame({'Feature': X_train_2.columns, 'Importance': xgb_model_stage2.feature_importances_})
feature_importance_2 = feature_importance_2.sort_values(by='Importance', ascending=False)
feature_importance_2 = feature_importance_2.head(20)
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_2)
plt.title('Feature Importance Stage 2')
plt.show()

Feature importance remains the same as in Stage 1, Nationalities from Bangladesh and India, followed by Centre Name.

In [ ]:
# XGBoost hyperparameter Stage 2
param_grid_2 = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5]
}
grid_search_2 = GridSearchCV(XGBClassifier(random_state=42), param_grid_2, cv=5, scoring='accuracy')
grid_search_2.fit(X_train_2, y_train_2)
best_params_2 = grid_search_2.best_params_
best_model_2 = grid_search_2.best_estimator_

In [ ]:
# Best parameters Stage 2
print("Best parameters Stage 2:", grid_search_2.best_params_)

In [ ]:
# Best model with the best parameters - Stage 2 Model 2
y_pred_2_model2 = best_model_2.predict(X_test_2)
print("Accuracy:", accuracy_score(y_test_2, y_pred_2_model2))
print("Confusion Matrix:", confusion_matrix(y_test_2, y_pred_2_model2))
print("Precision:", precision_score(y_test_2, y_pred_2_model2))
print("Recall:", recall_score(y_test_2, y_pred_2_model2))

cm_2 = confusion_matrix(y_test_2, y_pred_2_model2)
sns.heatmap(cm_2, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix Stage 2 - Model 2')
plt.show()

The Accuracy for the base model is 90.0%, and the Recall is 96.1%, where the precision is 92.4%. The Confusion Matrix showed 337 false positives. There are 164 false negatives (flagged incorrectly by the model).

In [ ]:
# AUC plot Stage 2 - Model 2
y_prob_2_model2 = best_model_2.predict_proba(X_test_2)[:,1]

# Calculate ROC curve
fpr_2, tpr_2, thresholds_2 = roc_curve(y_test_2, y_prob_2_model2)
roc_auc_2_model2 = auc(fpr_2, tpr_2)

# Plot ROC curve
plt.figure()
plt.plot(fpr_2, tpr_2, color='darkorange', lw=2, label='ROC curve (area = %0.1f)' % roc_auc_2_model2)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Stage 2 - Model 2')
plt.legend()
plt.show()

print("AUC Score:", roc_auc_2_model2)

No changes on the Accuracy results compared to Model 1.

In [ ]:
# Compared both XGBoost Models Stage 2
print("XGBoost Stage 2 - Model 1")
print("Accuracy:", accuracy_score(y_test_2, y_pred_xgb_stage2))
print("Precision:", precision_score(y_test_2, y_pred_xgb_stage2))
print("Recall:", recall_score(y_test_2, y_pred_xgb_stage2))
print("AUC:", roc_auc_xgb_stage2)

print("\nXGBoost Stage 2 - Model 2 (Tuned)")
print("Accuracy:", accuracy_score(y_test_2, y_pred_2_model2))
print("Precision:", precision_score(y_test_2, y_pred_2_model2))
print("Recall:", recall_score(y_test_2, y_pred_2_model2))
print("AUC:", roc_auc_2_model2)

In [ ]:
# Data for comparison Stage 2
metrics = ['Accuracy', 'Precision', 'Recall', 'AUC']
original_scores_2 = [
    accuracy_score(y_test_2, y_pred_xgb_stage2),
    precision_score(y_test_2, y_pred_xgb_stage2),
    recall_score(y_test_2, y_pred_xgb_stage2),
    roc_auc_xgb_stage2
]
tuned_scores_2 = [
    accuracy_score(y_test_2, y_pred_2_model2),
    precision_score(y_test_2, y_pred_2_model2),
    recall_score(y_test_2, y_pred_2_model2),
    roc_auc_2_model2
]

# Plot
x = range(len(metrics))
fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x, original_scores_2, width=0.4, label='Original', align='center')
bars2 = ax.bar([i + 0.4 for i in x], tuned_scores_2, width=0.4, label='Tuned', align='center')

# Add data labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks([i + 0.2 for i in x])
ax.set_xticklabels(metrics)
ax.set_title('XGBoost Stage 2: Original vs Tuned')
ax.set_ylabel('Score')
ax.set_ylim(0.8, 1.0)
ax.legend()
plt.show()

Results on both XGboost models are pretty similar to Recall (0962) and AUC (0962). It is slightly better fo Model 1 for Accuracy (0902) and Precision (0927).


Accuracy: Stage 1: 0.891 Stage 2: 0.902

Precision: Stage 1: 0.920 Stage 2: 0.927

Recall: Stage 1: 0.956 Stage 2: 0.962

AUC: Stage 1: 0.876 Stage 2: 0.903

Adding engagement data, the Authorised Absence Count column, improved the model AUC, which improved the most from 0.87 to 0.90. This confirms that engagement data helps predict dropout more accurately. However, improvements are modest, suggesting that the Stage 1 applicant data was already quite predictive.

## Neural Network

**Defining the functions**

In [ ]:
# Standardise the features Stage 2
scaler_2 = StandardScaler()
X_train_2_scaled = scaler_2.fit_transform(X_train_2)
X_val_2_scaled = scaler_2.transform(X_val_2)
X_test_2_scaled = scaler_2.transform(X_test_2)

In [ ]:
# Number of features Stage 2
num_features_stage2 = X_train_2_scaled.shape[1]

In [ ]:
# Print number of features Stage 2
print("Number of features Stage 2:", num_features_stage2)

In [ ]:
# One input layer of a number of neurons equal to the number of input features
model_nn_stage2 = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(X_train_2_scaled.shape[1],)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
# Binary cross entropy as loss
model_nn_stage2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

**Create the Neural Network and train the models**

In [ ]:
# Create and train models with Adam Stage 2
adam_stage2 = model_nn_stage2.fit(X_train_2_scaled, y_train_2, epochs=50, batch_size=32, validation_data=(X_val_2_scaled, y_val_2))

In [ ]:
# Loss curves for each epoch plotted Stage 2 Model 1
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(adam_stage2.history['loss'], label='Training Loss')
plt.plot(adam_stage2.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss Stage 2 - Model 1')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(adam_stage2.history['accuracy'], label='Training Accuracy')
plt.plot(adam_stage2.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy Stage 2 - Model 1')
plt.legend()

plt.tight_layout()
plt.show()

Model 1 clearly overfitted after epoch 30 for the validation loss.

**Function with dropout**

In [ ]:
# Function regularisation and dropout Stage 2
def build_model_stage2(optimizer):
    model_stage2 = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu',
                              input_shape=(X_train_2_scaled.shape[1],),
                              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.Dense(16, activation='relu',
                              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model_stage2.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model_stage2

In [ ]:
# Adam optimization and L2 regularisation Stage 2
adam_stage2_l2 = build_model_stage2(Adam(learning_rate=0.001))
adam_stage2_l2_history = adam_stage2_l2.fit(X_train_2_scaled, y_train_2, epochs=50, batch_size=32, validation_data=(X_val_2_scaled, y_val_2))

In [ ]:
# Sanity check Stage 2
print(np.isnan(X_train_2_scaled).sum())
print(np.isnan(X_test_2_scaled).sum())

In [ ]:
# Loss curves for each epoch plotted Stage 2 Model 2
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(adam_stage2_l2_history.history['loss'], label='Training Loss')
plt.plot(adam_stage2_l2_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss Stage 2 - Model 2')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(adam_stage2_l2_history.history['accuracy'], label='Training Accuracy')
plt.plot(adam_stage2_l2_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy Stage 2 - Model 2')
plt.legend()

plt.tight_layout()
plt.show()

Model 2 performed slightly better than Model 1, reducing the gap between training and Validation for both loss and accuracy.

**Early stopping**

In [ ]:
# Neural layers Stage 2
def build_model_adam_stage2_v1(optimizer):
    adam_stage2_l2_es = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(X_train_2_scaled.shape[1],), kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    adam_stage2_l2_es.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return adam_stage2_l2_es

# Early stopping Stage 2
early_stopping_stage2 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

# Pass it to the model
adam_stage2_l2_es = build_model_adam_stage2_v1(Adam(learning_rate=0.001))
adam_stage2_l2_es_history = adam_stage2_l2_es.fit(
    X_train_2_scaled, y_train_2, epochs=50, batch_size=32,
    validation_data=(X_val_2_scaled, y_val_2),
    callbacks=[early_stopping_stage2])

Training stopped on epoch 13.

In [ ]:
# Loss curves for each epoch plotted Stage 2 Model 3
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(adam_stage2_l2_es_history.history['loss'], label='Training Loss')
plt.plot(adam_stage2_l2_es_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss Stage 2 - Model 3')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(adam_stage2_l2_es_history.history['accuracy'], label='Training Accuracy')
plt.plot(adam_stage2_l2_es_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy Stage 2 - Model 3')
plt.legend()

plt.tight_layout()
plt.show()

Model 3 indicates there is no overfitting on the Training and Validation loss. Training and Validation accuracy get higher for Validation, but it stabilises at 0.8968 (89%) of the dataset. It was noticed small fluctuations but nothign major.


**Test one extra layer**

In [ ]:
# Test one extra layer Stage 2
def build_model_adam_stage2_v2(optimizer):
    adam_stage2_l2_es2 = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train_2_scaled.shape[1],), kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    adam_stage2_l2_es2.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return adam_stage2_l2_es2

early_stopping2_stage2 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

adam_stage2_l2_es2 = build_model_adam_stage2_v2(Adam(learning_rate=0.001))
adam_stage2_l2_es2_history = adam_stage2_l2_es2.fit(
    X_train_2_scaled, y_train_2, epochs=50, batch_size=32,
    validation_data=(X_val_2_scaled, y_val_2),
    callbacks=[early_stopping2_stage2])

With one extra layer the epochs stopped on 15 which means the Model learned until 14 epochs.

In [ ]:
# Loss curves for each epoch plotted Stage 2 - Model 4
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(adam_stage2_l2_es2_history.history['loss'], label='Training Loss')
plt.plot(adam_stage2_l2_es2_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss Stage 2 - Model 4')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(adam_stage2_l2_es2_history.history['accuracy'], label='Training Accuracy')
plt.plot(adam_stage2_l2_es2_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy Stage 2 - Model 4')
plt.legend()

plt.tight_layout()
plt.show()

Model 4 gets one extra layer, which indicates there is no overfitting on the Training and Validation loss as well. Training and Validation accuracy get higher for Validation, but it stabilises at 0.8963 (89%) of the dataset. It was noticed that there were small fluctuations, but nothing major.



**Test with RMSprop**

In [ ]:
# RMSprop with the same number of layers as model 3 - Stage 2
def build_model_rmsprop_stage2_v1(optimizer):
    rmsprop_stage2_l2_es = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(X_train_2_scaled.shape[1],), kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    rmsprop_stage2_l2_es.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return rmsprop_stage2_l2_es

early_stopping_rmsprop_stage2 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

rmsprop_stage2_l2_es = build_model_rmsprop_stage2_v1(RMSprop(learning_rate=0.001))
rmsprop_stage2_l2_es_history = rmsprop_stage2_l2_es.fit(
    X_train_2_scaled, y_train_2, epochs=50, batch_size=32,
    validation_data=(X_val_2_scaled, y_val_2),
    callbacks=[early_stopping_rmsprop_stage2])

Testing the model with RMSprop, it stopped after 7 epochs which just consolidates this optimizer is not the best for this dataset.

In [ ]:
# Loss curves for each epoch plotted Stage 2 - RMSprop - Model 5
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(rmsprop_stage2_l2_es_history.history['loss'], label='Training Loss')
plt.plot(rmsprop_stage2_l2_es_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss Stage 2 - RMSprop - Model 5')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(rmsprop_stage2_l2_es_history.history['accuracy'], label='Training Accuracy')
plt.plot(rmsprop_stage2_l2_es_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy Stage 2 - RMSprop - Model 5')
plt.legend()

plt.tight_layout()
plt.show()

Similar pattern as on Stage 1, using the same optimiser. The Neural Network didn't learn enough to find patterns. The Validation Loss seems to estruggle to learn from the trainign dataset.Training Accuracy remains lower in contrast to the Validation Accuracy.

**Models comparison**

In [ ]:
# Model Comparison - Stage 2
# XGBoost predictions - Model 1
y_pred_xgb_stage2 = xgb_model_stage2.predict(X_test_2)
y_prob_xgb_stage2 = xgb_model_stage2.predict_proba(X_test_2)[:,1]

# XGBoost predictions - Model 2 tuned
y_pred_xgb_tuned_stage2 = best_model_2.predict(X_test_2)
y_prob_xgb_tuned_stage2 = best_model_2.predict_proba(X_test_2)[:,1]

# Neural Network predictions - All 5 models
y_pred_nn_baseline_stage2 = (model_nn_stage2.predict(X_test_2_scaled) > 0.5).astype(int)
y_prob_nn_baseline_stage2 = model_nn_stage2.predict(X_test_2_scaled)

y_pred_nn_l2_stage2 = (adam_stage2_l2.predict(X_test_2_scaled) > 0.5).astype(int)
y_prob_nn_l2_stage2 = adam_stage2_l2.predict(X_test_2_scaled)

y_pred_nn_v1_stage2 = (adam_stage2_l2_es.predict(X_test_2_scaled) > 0.5).astype(int)
y_prob_nn_v1_stage2 = adam_stage2_l2_es.predict(X_test_2_scaled)

y_pred_nn_v2_stage2 = (adam_stage2_l2_es2.predict(X_test_2_scaled) > 0.5).astype(int)
y_prob_nn_v2_stage2 = adam_stage2_l2_es2.predict(X_test_2_scaled)

y_pred_nn_rmsprop_stage2 = (rmsprop_stage2_l2_es.predict(X_test_2_scaled) > 0.5).astype(int)
y_prob_nn_rmsprop_stage2 = rmsprop_stage2_l2_es.predict(X_test_2_scaled)

# All models dictionary
models_stage2 = {
    'XGBoost_stage2': (y_pred_xgb_stage2, y_prob_xgb_stage2),
    'XGBoost_Tuned_stage2': (y_pred_xgb_tuned_stage2, y_prob_xgb_tuned_stage2),
    'NN_Baseline_stage2': (y_pred_nn_baseline_stage2, y_prob_nn_baseline_stage2),
    'NN_L2_stage2': (y_pred_nn_l2_stage2, y_prob_nn_l2_stage2),
    'NN_Adam_32_16_stage2': (y_pred_nn_v1_stage2, y_prob_nn_v1_stage2),
    'NN_Adam_64_32_16_stage2': (y_pred_nn_v2_stage2, y_prob_nn_v2_stage2),
    'NN_RMSprop_32_16_stage2': (y_pred_nn_rmsprop_stage2, y_prob_nn_rmsprop_stage2)
}

for name, (y_pred, y_prob) in models_stage2.items():
    print(f"\n=== {name} ===")
    print("Accuracy:", accuracy_score(y_test_2, y_pred))
    print("Precision:", precision_score(y_test_2, y_pred))
    print("Recall:", recall_score(y_test_2, y_pred))
    print("AUC:", roc_auc_score(y_test_2, y_prob))
    print("Confusion Matrix:\n", confusion_matrix(y_test_2, y_pred))

In [ ]:
# Plot comparison of all models Stage 2 - Horizontal
metrics_names = ['Accuracy', 'Precision', 'Recall', 'AUC']

scores_stage2 = {}
for name, (y_pred, y_prob) in models_stage2.items():
    scores_stage2[name] = [
        accuracy_score(y_test_2, y_pred),
        precision_score(y_test_2, y_pred),
        recall_score(y_test_2, y_pred),
        roc_auc_score(y_test_2, y_prob)
    ]

fig, axes = plt.subplots(1, 4, figsize=(20, 6))

for idx, metric in enumerate(metrics_names):
    ax = axes[idx]
    model_names = list(scores_stage2.keys())
    scores = [scores_stage2[m][idx] for m in model_names]

    bars = ax.barh(model_names, scores, color='#d94801')

    for bar, score in zip(bars, scores):
        ax.text(bar.get_width() - 0.002, bar.get_y() + bar.get_height()/2,
                f'{score:.3f}', ha='right', va='center',
                fontsize=8, color='white', fontweight='bold')

    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_xlim(0.8, 1.0)
    ax.set_xlabel('Score')
    ax.grid(True, alpha=0.3)

plt.suptitle('Model Comparison - Stage 2', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In the two methodologies used, again , they had mixed results as the XGBoost Model 1 reached higher accuracy (0.902), had mixed results as the XGBoost Model 1 reached higher accuracy (0.902) but AUC got the same results for both XGBoost models (0.903). The Precision on Model 3 (0.931) was higher in the Neural Network methodology. As mentioned before on Stage 1, the data is imbalanced, and the Recall would be the primary metric for Model 1 and 2 (0.962) for the XGboost reached better results.

# Comparisons between Stage 1 & 2

In [ ]:
y_pred_nn_baseline_stage2 = (model_nn_stage2.predict(X_test_2_scaled) > 0.5).astype(int)
y_prob_nn_baseline_stage2 = model_nn_stage2.predict(X_test_2_scaled)

y_pred_nn_l2_stage2 = (adam_stage2_l2.predict(X_test_2_scaled) > 0.5).astype(int)
y_prob_nn_l2_stage2 = adam_stage2_l2.predict(X_test_2_scaled)

In [ ]:
# Visualization - Stage 1 vs Stage 2 All Models
metrics_names = ['Accuracy', 'Precision', 'Recall', 'AUC']

# Calculate scores for all models
stage1_scores = {
    'XGBoost': [accuracy_score(y_test_1, y_pred_xgb_stage1), precision_score(y_test_1, y_pred_xgb_stage1), recall_score(y_test_1, y_pred_xgb_stage1), roc_auc_score(y_test_1, y_prob_xgb_stage1)],
    'XGBoost_Tuned': [accuracy_score(y_test_1, y_pred_xgb_tuned_stage1), precision_score(y_test_1, y_pred_xgb_tuned_stage1), recall_score(y_test_1, y_pred_xgb_tuned_stage1), roc_auc_score(y_test_1, y_prob_xgb_tuned_stage1)],
    'NN_Baseline': [accuracy_score(y_test_1, y_pred_nn_baseline_stage1), precision_score(y_test_1, y_pred_nn_baseline_stage1), recall_score(y_test_1, y_pred_nn_baseline_stage1), roc_auc_score(y_test_1, y_prob_nn_baseline_stage1)],
    'NN_L2': [accuracy_score(y_test_1, y_pred_nn_l2_stage1), precision_score(y_test_1, y_pred_nn_l2_stage1), recall_score(y_test_1, y_pred_nn_l2_stage1), roc_auc_score(y_test_1, y_prob_nn_l2_stage1)],
    'NN_Adam_32_16': [accuracy_score(y_test_1, y_pred_nn_v1_stage1), precision_score(y_test_1, y_pred_nn_v1_stage1), recall_score(y_test_1, y_pred_nn_v1_stage1), roc_auc_score(y_test_1, y_prob_nn_v1_stage1)],
    'NN_Adam_64_32_16': [accuracy_score(y_test_1, y_pred_nn_v2_stage1), precision_score(y_test_1, y_pred_nn_v2_stage1), recall_score(y_test_1, y_pred_nn_v2_stage1), roc_auc_score(y_test_1, y_prob_nn_v2_stage1)],
    'NN_RMSprop_32_16': [accuracy_score(y_test_1, y_pred_nn_rmsprop_stage1), precision_score(y_test_1, y_pred_nn_rmsprop_stage1), recall_score(y_test_1, y_pred_nn_rmsprop_stage1), roc_auc_score(y_test_1, y_prob_nn_rmsprop_stage1)]
}

stage2_scores = {
    'XGBoost': [accuracy_score(y_test_2, y_pred_xgb_stage2), precision_score(y_test_2, y_pred_xgb_stage2), recall_score(y_test_2, y_pred_xgb_stage2), roc_auc_score(y_test_2, y_prob_xgb_stage2)],
    'XGBoost_Tuned': [accuracy_score(y_test_2, y_pred_xgb_tuned_stage2), precision_score(y_test_2, y_pred_xgb_tuned_stage2), recall_score(y_test_2, y_pred_xgb_tuned_stage2), roc_auc_score(y_test_2, y_prob_xgb_tuned_stage2)],
    'NN_Baseline': [accuracy_score(y_test_2, y_pred_nn_baseline_stage2), precision_score(y_test_2, y_pred_nn_baseline_stage2), recall_score(y_test_2, y_pred_nn_baseline_stage2), roc_auc_score(y_test_2, y_prob_nn_baseline_stage2)],
    'NN_L2': [accuracy_score(y_test_2, y_pred_nn_l2_stage2), precision_score(y_test_2, y_pred_nn_l2_stage2), recall_score(y_test_2, y_pred_nn_l2_stage2), roc_auc_score(y_test_2, y_prob_nn_l2_stage2)],
    'NN_Adam_32_16': [accuracy_score(y_test_2, y_pred_nn_v1_stage2), precision_score(y_test_2, y_pred_nn_v1_stage2), recall_score(y_test_2, y_pred_nn_v1_stage2), roc_auc_score(y_test_2, y_prob_nn_v1_stage2)],
    'NN_Adam_64_32_16': [accuracy_score(y_test_2, y_pred_nn_v2_stage2), precision_score(y_test_2, y_pred_nn_v2_stage2), recall_score(y_test_2, y_pred_nn_v2_stage2), roc_auc_score(y_test_2, y_prob_nn_v2_stage2)],
    'NN_RMSprop_32_16': [accuracy_score(y_test_2, y_pred_nn_rmsprop_stage2), precision_score(y_test_2, y_pred_nn_rmsprop_stage2), recall_score(y_test_2, y_pred_nn_rmsprop_stage2), roc_auc_score(y_test_2, y_prob_nn_rmsprop_stage2)]
}

model_names = list(stage1_scores.keys())

fig, axes = plt.subplots(1, 4, figsize=(24, 8))

for idx, metric in enumerate(metrics_names):
    ax = axes[idx]

    stage1_vals = [stage1_scores[m][idx] for m in model_names]
    stage2_vals = [stage2_scores[m][idx] for m in model_names]

    y = np.arange(len(model_names))
    height = 0.35

    bars1 = ax.barh(y - height/2, stage1_vals, height, label='Stage 1', color='#2171b5')
    bars2 = ax.barh(y + height/2, stage2_vals, height, label='Stage 2', color='#238b45')

    # Add data labels
    for bar in bars1:
        ax.text(bar.get_width() - 0.002, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.3f}', ha='right', va='center',
                fontsize=7, color='white', fontweight='bold')

    for bar in bars2:
        ax.text(bar.get_width() - 0.002, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.3f}', ha='right', va='center',
                fontsize=7, color='white', fontweight='bold')

    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_yticks(y)
    ax.set_yticklabels(model_names, fontsize=8)
    ax.set_xlim(0.8, 1.05)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Stage 1 vs Stage 2 - All Models Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Amongst Stage 1 and 2, it was noticed that Stage 2 performed better using the same parameters and the same target variable.

The Accuracy results were better with XGBoost Model 1, reaching (0.902) across the dataset. For Precision, the Model 3 resulted in (0.931). AUC (0.903) and Recall (0.962) performed better with XGBoost Models 1 and 2.  

Overall, Stage 2 performed better across all 4 metrics in most of the models tested. Accuracy and Precision had better results with Neural Network methods whereas AUC and Recall using XGBoost methodologies.

The potential reasons for the mixed results it can be the extra feature on stage 2, XGBoost is expected to perform better in tabular data and also imbalance data but also due to the additional feature added to the dataset. However we could only confirm it with the domain expert.

# Stage 3 data: Academic performance data

## EDA

In [ ]:
# File URL
stage3 = "<path-to-dataset>"

In [ ]:
# Import the necessary libraries.
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam, RMSprop
import matplotlib.pyplot as plt
import seaborn as sns
from keras.callbacks import EarlyStopping
from keras.regularizers import l2
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.metrics import roc_curve, auc
import tensorflow as tf

In [ ]:
# Data exploration
stage3 = pd.read_csv(stage3)
stage3.head()

In [ ]:
stage3.info()

In [ ]:
stage3.shape

In [ ]:
# Missing values
stage2.isnull().sum()

## Pre-processing instructions

- Remove any columns not useful in the analysis (LearnerCode).
- Remove columns with categorical features with high cardinality (use >200 unique values, as a guideline for this data set).
- Remove columns with >50% data missing.
- Perform ordinal encoding for ordinal data.
- Perform one-hot encoding for all other categorical data.
- Choose how to engage with rows that have missing values, which can be done in one of two ways for this project:
  *   Impute the rows with appropriate values.
  *   Remove rows with missing values but ONLY in cases where rows with missing values are minimal: <2% of the overall data.






In [ ]:
# Remove any columns not useful in the analysis (LearnerCode)
stage3 = stage3.drop('LearnerCode', axis=1)

In [ ]:
# Transform DOB into Age before drop it
stage3['DateofBirth'] = pd.to_datetime(stage3['DateofBirth'])
stage3['Age'] = 2026 - stage3['DateofBirth'].dt.year

In [ ]:
# Sanity check on Age column
stage3['Age'].describe()

In [ ]:
# Remove columns with categorical features with high cardinality (use >200 unique values, as a guideline for this data set)
high_cardinality_3 = stage3.nunique()[stage3.nunique() > 200].index
stage3.drop(columns=high_cardinality_3, inplace=True)

In [ ]:
stage2.dtypes

In [ ]:
# Remove columns with > 50% data missing.
missing_percentage_3 = stage3.isnull().mean() * 100
missing_columns_3 = missing_percentage_3[missing_percentage_3 > 50].index
stage3 = stage3.drop(columns=missing_columns_3)

In [ ]:
stage3.dtypes

In [ ]:
stage3.shape

In [ ]:
stage3.nunique()

In [ ]:
print(stage3['BookingType'].unique())
print(stage3['Gender'].unique())
print(stage3['CompletedCourse'].unique())
print(stage3['IsFirstIntake'].unique())

In [ ]:
# Perform ordinal encoding for ordinal data
# Binary mapping for binary categorical columns
binary_mapping3 = {'Agent': 1, 'Direct': 0, 'Female': 1, 'Male': 0, 'Yes': 1, 'No': 0}

stage3['BookingType'] = stage3['BookingType'].map(binary_mapping2)
stage3['Gender'] = stage3['Gender'].map(binary_mapping2)
stage3['IsFirstIntake'] = stage3['IsFirstIntake'].astype(int)
stage3['CompletedCourse'] = stage3['CompletedCourse'].map(binary_mapping2)

In [ ]:
# Sanity check on the binary data applied
print(stage3['BookingType'].unique())
print(stage3['Gender'].unique())
print(stage3['IsFirstIntake'].unique())
print(stage3['CompletedCourse'].unique())

In [ ]:
# Checking the columns that should be enconding for categorical data.
stage3.select_dtypes(include=['object']).columns

In [ ]:
# Perform one-hot encoding for all other categorical data.
stage3 = pd.get_dummies(stage3, columns=['CentreName', 'LeadSource', 'Nationality', 'CourseLevel', 'CourseName', 'ProgressionUniversity'])

In [ ]:
stage3.head()

In [ ]:
stage3.dtypes

Some columns becomes boolean and it will be reverted.

In [ ]:
# Convert bool columns to int
stage3 = stage3.astype({col: int for col in stage3.select_dtypes(bool).columns})

In [ ]:
print(stage3.dtypes.value_counts())

In [ ]:
# Check percentage of missing values for each column
missing_percentage_3 = stage3.isnull().sum() / len(stage3) * 100
print(missing_percentage_3[missing_percentage_3 > 0])

The Authorised Absence Count feature was added on Stage 2 and to keep the results consistent and be able to compared with them I will keep it. In adition to that I will keep the other ones since this will enrich the analysis and help the MLs models learn with new parameters.

In [ ]:
for col in ['AssessedModules', 'PassedModules', 'FailedModules', 'AuthorisedAbsenceCount']:
    stage3[col] = stage3[col].fillna(stage3[col].mean())

In [ ]:
print(stage3.isnull().sum().sum())

In [ ]:
stage3.dtypes.value_counts()

## Define Features and target variable

In [ ]:
# Define features and target varible
y = stage3['CompletedCourse']
X = stage3.drop('CompletedCourse', axis=1)

In [ ]:
sns.countplot(x='CompletedCourse', data=stage3)
plt.title('Target Variable Distribution Stage 3')
plt.show()

As in Stages 1 and 2, Stage 3 also has imbalanced data.

## Train test

In [ ]:
# 80/20 split
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 10% validation split
X_train_3, X_val_3, y_train_3, y_val_3 = train_test_split(X_train_3, y_train_3, test_size=0.1, random_state=42)

## XGBoost

In [ ]:
# XGBoost Stage 3
xgb_model_stage3 = XGBClassifier(random_state=42)
xgb_model_stage3.fit(X_train_3, y_train_3)

In [ ]:
# Model evaluation Stage 3
y_pred_xgb_stage3 = xgb_model_stage3.predict(X_test_3)
print("Accuracy:", accuracy_score(y_test_3, y_pred_xgb_stage3))
print("Confusion Matrix:", confusion_matrix(y_test_3, y_pred_xgb_stage3))
print("Precision:", precision_score(y_test_3, y_pred_xgb_stage3))
print("Recall:", recall_score(y_test_3, y_pred_xgb_stage3))

cm = confusion_matrix(y_test_3, y_pred_xgb_stage3)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix Stage 3 - Model 1')
plt.show()

Accuracy, Precision, and Recall are 99%, which is suspicious and should be investigated for potential data leakage.

In [ ]:
print(stage3['CompletedCourse'].value_counts())

In [ ]:
# AUC plot Stage 3
y_prob_xgb_stage3 = xgb_model_stage3.predict_proba(X_test_3)[:,1]

# Calculate ROC curve
fpr_3, tpr_3, thresholds_3 = roc_curve(y_test_3, y_prob_xgb_stage3)
roc_auc_xgb_stage3 = auc(fpr_3, tpr_3)

# Plot ROC curve
plt.figure()
plt.plot(fpr_3, tpr_3, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc_xgb_stage3)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Stage 3 - Model 1')
plt.legend()
plt.show()

print("AUC Score:", roc_auc_xgb_stage3)

The true positive rate is pretty high, as the curve of the ROC indicates 1.00. Which initially looks good but due to the new columns on the dataset it can indicate that the de dataset received data that is similar to the target varible and the model just concluded that everyone dropped out because the course finished.

In [ ]:
# Feature importance Stage 3
feature_importance_3 = pd.DataFrame({'Feature': X_train_3.columns, 'Importance': xgb_model_stage3.feature_importances_})
feature_importance_3 = feature_importance_3.sort_values(by='Importance', ascending=False)
feature_importance_3 = feature_importance_3.head(20)
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_3)
plt.title('Feature Importance Stage 3')
plt.show()

The top three features, compared to Stages 1 and 2, were switched from Nationalities to Failed Modules, Assessed Modules, and finally to a specific Centre Name. Since the analyst does not have access to the domain expert, it is not possible to determine whether this is accurate or to obtain more details about how the data was collected or how it aligns with their business.

In [ ]:
# Correlation between the target variable with the new features
corr = stage3[['CompletedCourse', 'AssessedModules',
               'PassedModules', 'FailedModules']].corr()
print(corr['CompletedCourse'])

None of the new features has high correlation with the target varible but the combination of academic performance features gives the model too much information specially because XGBoost can overfitting. Since the domain expert is not available for further discussion I will tuned the model and check for any other pattern.

In [ ]:
# XGBoost hyperparameter Stage 3
param_grid_3 = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5]
}
grid_search_3 = GridSearchCV(XGBClassifier(random_state=42), param_grid_3, cv=5, scoring='accuracy')
grid_search_3.fit(X_train_3, y_train_3)
best_params_3 = grid_search_3.best_params_
best_model_3 = grid_search_3.best_estimator_

In [ ]:
# Best parameters
print("Best parameters Stage 3:", grid_search_3.best_params_)

In [ ]:
# Best model with the best parameters tuned - Stage 3
y_pred_3_tuned = best_model_3.predict(X_test_3)
print("Accuracy:", accuracy_score(y_test_3, y_pred_3_tuned))
print("Confusion Matrix:", confusion_matrix(y_test_3, y_pred_3_tuned))
print("Precision:", precision_score(y_test_3, y_pred_3_tuned))
print("Recall:", recall_score(y_test_3, y_pred_3_tuned))

cm_3 = confusion_matrix(y_test_3, y_pred_3_tuned)
sns.heatmap(cm_3, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix Stage 3 - Model 2')
plt.show()

Accuracy, Precision, and Recall remained at 99%. Slitgly changes on the Confusion matrix.

In [ ]:
# Compared both XGBoost Models Stage 3
print("XGBoost Stage 3 - Model 1")
print("Accuracy:", accuracy_score(y_test_3, y_pred_xgb_stage3))
print("Precision:", precision_score(y_test_3, y_pred_xgb_stage3))
print("Recall:", recall_score(y_test_3, y_pred_xgb_stage3))
print("AUC:", roc_auc_xgb_stage3)

# Calculate AUC for tuned model
y_prob_xgb_tuned_stage3 = best_model_3.predict_proba(X_test_3)[:,1]
roc_auc_xgb_tuned_stage3 = roc_auc_score(y_test_3, y_prob_xgb_tuned_stage3)

print("\nXGBoost Stage 3 - Model 2 (Tuned)")
print("Accuracy:", accuracy_score(y_test_3, y_pred_3_tuned))
print("Precision:", precision_score(y_test_3, y_pred_3_tuned))
print("Recall:", recall_score(y_test_3, y_pred_3_tuned))
print("AUC:", roc_auc_xgb_tuned_stage3)

The XGBoost results still remains quite suspicious as most of the metrics for both modules kept at 99%.

## Neural Network

In [ ]:
# Standardise the features Stage 3
scaler_3 = StandardScaler()
X_train_3_scaled = scaler_3.fit_transform(X_train_3)
X_val_3_scaled = scaler_3.transform(X_val_3)
X_test_3_scaled = scaler_3.transform(X_test_3)

In [ ]:
# Number of features Stage 3
num_features_stage3 = X_train_3_scaled.shape[1]

In [ ]:
# Print number of features Stage 3
print("Number of features Stage 3:", num_features_stage3)

In [ ]:
# One input layer of a number of neurons equal to the number of input features
model_nn_stage3 = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(X_train_3_scaled.shape[1],)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
# Binary cross entropy as loss
model_nn_stage3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

**Create Neural Network and train the models**

In [ ]:
# Create and train models with Adam Stage 3
adam_stage3 = model_nn_stage3.fit(X_train_3_scaled, y_train_3, epochs=50, batch_size=32, validation_data=(X_val_3_scaled, y_val_3))

In [ ]:
# Loss curves for each epoch plotted Stage 3 - Model 1
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(adam_stage3.history['loss'], label='Training Loss')
plt.plot(adam_stage3.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss Stage 3 - Model 1')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(adam_stage3.history['accuracy'], label='Training Accuracy')
plt.plot(adam_stage3.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy Stage 3 - Model 1')
plt.legend()

plt.tight_layout()
plt.show()

Model 1 overfitted after epoch 33 for the validation loss.

**Function with dropout**

In [ ]:
# Function regularisation and dropout Stage 3
def build_model_stage3(optimizer):
    model_stage3 = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu',
                              input_shape=(X_train_3_scaled.shape[1],),
                              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.Dense(16, activation='relu',
                              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model_stage3.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model_stage3

In [ ]:
# Adam optimization and L2 regularisation Stage 3
adam_stage3_l2 = build_model_stage3(Adam(learning_rate=0.001))
adam_stage3_l2_history = adam_stage3_l2.fit(X_train_3_scaled, y_train_3, epochs=50, batch_size=32, validation_data=(X_val_3_scaled, y_val_3))

In [ ]:
# Sanity check Stage 3
print(np.isnan(X_train_3_scaled).sum())
print(np.isnan(X_test_3_scaled).sum())

In [ ]:
# Loss curves for each epoch plotted Stage 3 - Model 2
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(adam_stage3_l2_history.history['loss'], label='Training Loss')
plt.plot(adam_stage3_l2_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss Stage 3 - Model 2')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(adam_stage3_l2_history.history['accuracy'], label='Training Accuracy')
plt.plot(adam_stage3_l2_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy Stage 3 - Model 2')
plt.legend()

plt.tight_layout()
plt.show()

Training Loss started very high and decreased, while Validation loss remained unstable throughout training.

Training and Validation accuracy remain high at 0.9840 (98%) on the dataset. Those results may indicate data leakage, but the domain expert would need to confirm it.

**Early stopping**

In [ ]:
# Neural layers Stage 3 Model 3
def build_model_adam_stage3_v1(optimizer):
    adam_stage3_l2_es = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(X_train_3_scaled.shape[1],), kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    adam_stage3_l2_es.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return adam_stage3_l2_es

# Early stopping Stage 3
early_stopping_stage3 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

# Pass it to the model
adam_stage3_l2_es = build_model_adam_stage3_v1(Adam(learning_rate=0.001))
adam_stage3_l2_es_history = adam_stage3_l2_es.fit(
    X_train_3_scaled, y_train_3, epochs=50, batch_size=32,
    validation_data=(X_val_3_scaled, y_val_3),
    callbacks=[early_stopping_stage3])

With early stopping, the NN stopped on 20 epochs.

In [ ]:
# Loss curves for each epoch plotted Stage 3 Model 3
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(adam_stage3_l2_es_history.history['loss'], label='Training Loss')
plt.plot(adam_stage3_l2_es_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss Stage 3 - Model 3')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(adam_stage3_l2_es_history.history['accuracy'], label='Training Accuracy')
plt.plot(adam_stage3_l2_es_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy Stage 3 - Model 3')
plt.legend()

plt.tight_layout()
plt.show()

Training Loss started very high and decreased at the end of the training. Validation Loss was a bit more stable.

Training and Validation Accuracy remain high at 0.9786 (97%) on the dataset.

**Test one extra layer**

In [ ]:
# Add 64 layer as test to Adam optimizer - Stage 3
def build_model_adam_stage3_v2(optimizer):
    adam_stage3_l2_es2 = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train_3_scaled.shape[1],), kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    adam_stage3_l2_es2.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return adam_stage3_l2_es2

early_stopping2_stage3 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

adam_stage3_l2_es2 = build_model_adam_stage3_v2(Adam(learning_rate=0.001))
adam_stage3_l2_es2_history = adam_stage3_l2_es2.fit(
    X_train_3_scaled, y_train_3, epochs=50, batch_size=32,
    validation_data=(X_val_3_scaled, y_val_3),
    callbacks=[early_stopping2_stage3])

With one extra dense layer (64), the NN stopped on 31 epochs.

In [ ]:
# Loss curves for each epoch plotted Stage 3 - Model 4
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(adam_stage3_l2_es2_history.history['loss'], label='Training Loss')
plt.plot(adam_stage3_l2_es2_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss Stage 3 - Model 4')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(adam_stage3_l2_es2_history.history['accuracy'], label='Training Accuracy')
plt.plot(adam_stage3_l2_es2_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy Stage 3 - Model 4')
plt.legend()

plt.tight_layout()
plt.show()

Similar results with the Training and Validation Loss and the previous Model.
Training and Validation Loss keep high 0.9805.

**Test with RMSprop**

In [ ]:
# RMSprop with the same number of layers stage 3 - Model 5
def build_model_rmsprop_stage3_v1(optimizer):
    rmsprop_stage3_l2_es = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(X_train_3_scaled.shape[1],), kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    rmsprop_stage3_l2_es.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return rmsprop_stage3_l2_es

early_stopping_rmsprop_stage3 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

rmsprop_stage3_l2_es = build_model_rmsprop_stage3_v1(RMSprop(learning_rate=0.001))
rmsprop_stage3_l2_es_history = rmsprop_stage3_l2_es.fit(
    X_train_3_scaled, y_train_3, epochs=50, batch_size=32,
    validation_data=(X_val_3_scaled, y_val_3),
    callbacks=[early_stopping_rmsprop_stage3])

In [ ]:
# Loss curves for each epoch plotted Stage 3 RMSprop - Model 5
plt.figure(figsize=(12, 6))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(rmsprop_stage3_l2_es_history.history['loss'], label='Training Loss')
plt.plot(rmsprop_stage3_l2_es_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss Stage 3 - RMSprop - Model 5')
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(rmsprop_stage3_l2_es_history.history['accuracy'], label='Training Accuracy')
plt.plot(rmsprop_stage3_l2_es_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy Stage 3 - RMSprop - Model 5')
plt.legend()

plt.tight_layout()
plt.show()

**Comparisons between the models on Stage 3**

In [ ]:
# Model Comparison - Stage 3
# XGBoost predictions
y_pred_xgb_stage3 = xgb_model_stage3.predict(X_test_3)
y_prob_xgb_stage3 = xgb_model_stage3.predict_proba(X_test_3)[:,1]

y_pred_xgb_tuned_stage3 = best_model_3.predict(X_test_3)
y_prob_xgb_tuned_stage3 = best_model_3.predict_proba(X_test_3)[:,1]

# Neural Network predictions - All 5 models
y_pred_nn_baseline_stage3 = (model_nn_stage3.predict(X_test_3_scaled) > 0.5).astype(int)
y_prob_nn_baseline_stage3 = model_nn_stage3.predict(X_test_3_scaled)

y_pred_nn_l2_stage3 = (adam_stage3_l2.predict(X_test_3_scaled) > 0.5).astype(int)
y_prob_nn_l2_stage3 = adam_stage3_l2.predict(X_test_3_scaled)

y_pred_nn_v1_stage3 = (adam_stage3_l2_es.predict(X_test_3_scaled) > 0.5).astype(int)
y_prob_nn_v1_stage3 = adam_stage3_l2_es.predict(X_test_3_scaled)

y_pred_nn_v2_stage3 = (adam_stage3_l2_es2.predict(X_test_3_scaled) > 0.5).astype(int)
y_prob_nn_v2_stage3 = adam_stage3_l2_es2.predict(X_test_3_scaled)

y_pred_nn_rmsprop_stage3 = (rmsprop_stage3_l2_es.predict(X_test_3_scaled) > 0.5).astype(int)
y_prob_nn_rmsprop_stage3 = rmsprop_stage3_l2_es.predict(X_test_3_scaled)

# All models dictionary
models_stage3 = {
    'XGBoost_stage3': (y_pred_xgb_stage3, y_prob_xgb_stage3),
    'XGBoost_Tuned_stage3': (y_pred_xgb_tuned_stage3, y_prob_xgb_tuned_stage3),
    'NN_Baseline_stage3': (y_pred_nn_baseline_stage3, y_prob_nn_baseline_stage3),
    'NN_L2_stage3': (y_pred_nn_l2_stage3, y_prob_nn_l2_stage3),
    'NN_Adam_32_16_stage3': (y_pred_nn_v1_stage3, y_prob_nn_v1_stage3),
    'NN_Adam_64_32_16_stage3': (y_pred_nn_v2_stage3, y_prob_nn_v2_stage3),
    'NN_RMSprop_32_16_stage3': (y_pred_nn_rmsprop_stage3, y_prob_nn_rmsprop_stage3)
}

for name, (y_pred, y_prob) in models_stage3.items():
    print(f"\n=== {name} ===")
    print("Accuracy:", accuracy_score(y_test_3, y_pred))
    print("Precision:", precision_score(y_test_3, y_pred))
    print("Recall:", recall_score(y_test_3, y_pred))
    print("AUC:", roc_auc_score(y_test_3, y_prob))
    print("Confusion Matrix:\n", confusion_matrix(y_test_3, y_pred))

In [ ]:
# Plot comparison of all models Stage 3 - Horizontal
metrics_names = ['Accuracy', 'Precision', 'Recall', 'AUC']

# Calculate scores for each model
scores_stage3 = {}
for name, (y_pred, y_prob) in models_stage3.items():
    scores_stage3[name] = [
        accuracy_score(y_test_3, y_pred),
        precision_score(y_test_3, y_pred),
        recall_score(y_test_3, y_pred),
        roc_auc_score(y_test_3, y_prob)
    ]

fig, axes = plt.subplots(1, 4, figsize=(20, 6))

for idx, metric in enumerate(metrics_names):
    ax = axes[idx]
    model_names = list(scores_stage3.keys())
    scores = [scores_stage3[m][idx] for m in model_names]

    bars = ax.barh(model_names, scores, color='#2171b5')

    # Add data labels
    for bar, score in zip(bars, scores):
        ax.text(bar.get_width() - 0.002, bar.get_y() + bar.get_height()/2,
                f'{score:.3f}', ha='right', va='center',
                fontsize=8, color='white', fontweight='bold')

    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_xlim(0.8, 1.0)
    ax.set_xlabel('Score')
    ax.grid(True, alpha=0.3)

plt.suptitle('Model Comparison - Stage 3', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

All the metrics receive high scores, indicating that the Stage 3 data may contain a leaking outcome feature, preventing the model from learning properly. With stakeholder and domain expert input, it would be beneficial to review all the models.

# Comparisons between Stage 2 & 3

In [ ]:
# Visualisation - Stage 2 vs Stage 3 All Models
metrics_names = ['Accuracy', 'Precision', 'Recall', 'AUC']

# Calculate scores for all models
stage2_scores = {
    'XGBoost': [accuracy_score(y_test_2, y_pred_xgb_stage2), precision_score(y_test_2, y_pred_xgb_stage2), recall_score(y_test_2, y_pred_xgb_stage2), roc_auc_score(y_test_2, y_prob_xgb_stage2)],
    'XGBoost_Tuned': [accuracy_score(y_test_2, y_pred_xgb_tuned_stage2), precision_score(y_test_2, y_pred_xgb_tuned_stage2), recall_score(y_test_2, y_pred_xgb_tuned_stage2), roc_auc_score(y_test_2, y_prob_xgb_tuned_stage2)],
    'NN_Baseline': [accuracy_score(y_test_2, y_pred_nn_baseline_stage2), precision_score(y_test_2, y_pred_nn_baseline_stage2), recall_score(y_test_2, y_pred_nn_baseline_stage2), roc_auc_score(y_test_2, y_prob_nn_baseline_stage2)],
    'NN_L2': [accuracy_score(y_test_2, y_pred_nn_l2_stage2), precision_score(y_test_2, y_pred_nn_l2_stage2), recall_score(y_test_2, y_pred_nn_l2_stage2), roc_auc_score(y_test_2, y_prob_nn_l2_stage2)],
    'NN_Adam_32_16': [accuracy_score(y_test_2, y_pred_nn_v1_stage2), precision_score(y_test_2, y_pred_nn_v1_stage2), recall_score(y_test_2, y_pred_nn_v1_stage2), roc_auc_score(y_test_2, y_prob_nn_v1_stage2)],
    'NN_Adam_64_32_16': [accuracy_score(y_test_2, y_pred_nn_v2_stage2), precision_score(y_test_2, y_pred_nn_v2_stage2), recall_score(y_test_2, y_pred_nn_v2_stage2), roc_auc_score(y_test_2, y_prob_nn_v2_stage2)],
    'NN_RMSprop_32_16': [accuracy_score(y_test_2, y_pred_nn_rmsprop_stage2), precision_score(y_test_2, y_pred_nn_rmsprop_stage2), recall_score(y_test_2, y_pred_nn_rmsprop_stage2), roc_auc_score(y_test_2, y_prob_nn_rmsprop_stage2)]
}

stage3_scores = {
    'XGBoost': [accuracy_score(y_test_3, y_pred_xgb_stage3), precision_score(y_test_3, y_pred_xgb_stage3), recall_score(y_test_3, y_pred_xgb_stage3), roc_auc_score(y_test_3, y_prob_xgb_stage3)],
    'XGBoost_Tuned': [accuracy_score(y_test_3, y_pred_xgb_tuned_stage3), precision_score(y_test_3, y_pred_xgb_tuned_stage3), recall_score(y_test_3, y_pred_xgb_tuned_stage3), roc_auc_score(y_test_3, y_prob_xgb_tuned_stage3)],
    'NN_Baseline': [accuracy_score(y_test_3, y_pred_nn_baseline_stage3), precision_score(y_test_3, y_pred_nn_baseline_stage3), recall_score(y_test_3, y_pred_nn_baseline_stage3), roc_auc_score(y_test_3, y_prob_nn_baseline_stage3)],
    'NN_L2': [accuracy_score(y_test_3, y_pred_nn_l2_stage3), precision_score(y_test_3, y_pred_nn_l2_stage3), recall_score(y_test_3, y_pred_nn_l2_stage3), roc_auc_score(y_test_3, y_prob_nn_l2_stage3)],
    'NN_Adam_32_16': [accuracy_score(y_test_3, y_pred_nn_v1_stage3), precision_score(y_test_3, y_pred_nn_v1_stage3), recall_score(y_test_3, y_pred_nn_v1_stage3), roc_auc_score(y_test_3, y_prob_nn_v1_stage3)],
    'NN_Adam_64_32_16': [accuracy_score(y_test_3, y_pred_nn_v2_stage3), precision_score(y_test_3, y_pred_nn_v2_stage3), recall_score(y_test_3, y_pred_nn_v2_stage3), roc_auc_score(y_test_3, y_prob_nn_v2_stage3)],
    'NN_RMSprop_32_16': [accuracy_score(y_test_3, y_pred_nn_rmsprop_stage3), precision_score(y_test_3, y_pred_nn_rmsprop_stage3), recall_score(y_test_3, y_pred_nn_rmsprop_stage3), roc_auc_score(y_test_3, y_prob_nn_rmsprop_stage3)]
}

model_names = list(stage2_scores.keys())

fig, axes = plt.subplots(1, 4, figsize=(24, 8))

for idx, metric in enumerate(metrics_names):
    ax = axes[idx]

    stage2_vals = [stage2_scores[m][idx] for m in model_names]
    stage3_vals = [stage3_scores[m][idx] for m in model_names]

    y = np.arange(len(model_names))
    height = 0.35

    bars1 = ax.barh(y - height/2, stage2_vals, height, label='Stage 2', color='#2171b5')
    bars2 = ax.barh(y + height/2, stage3_vals, height, label='Stage 3', color='#238b45')

    # Add data labels
    for bar in bars1:
        ax.text(bar.get_width() - 0.002, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.3f}', ha='right', va='center',
                fontsize=7, color='white', fontweight='bold')

    for bar in bars2:
        ax.text(bar.get_width() - 0.002, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.3f}', ha='right', va='center',
                fontsize=7, color='white', fontweight='bold')

    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_yticks(y)
    ax.set_yticklabels(model_names, fontsize=8)
    ax.set_xlim(0.8, 1.05)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Stage 2 vs Stage 3 - All Models Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Comparison between Stage 2 and 3 performances should be interpreted with caution, since the data issues in Stage 3 were not properly addressed.

The first impression suggests that Stage 3 overcomes Stage 2, but in reality, this may be due to data leakage. More assumptions about removing the three new features could yield a different outcome. However, to make any further decision, the domain expert and other stakeholders should be contacted for their input on the dataset, and the outcome should be reviewed afterwards.

# Declaration
By submitting your project, you indicate that the work is your own and has been created with academic integrity. Refer to the **Cambridge plagiarism regulations**.
